In [9]:
import sys
from pathlib import Path
from uncertainties import ufloat
import country_converter as coco
import matplotlib.pyplot as plt
import re
import pandas as pd
import math
import numpy as np
from uncertainties import ufloat, nominal_value, std_dev

BASE_DIR = Path.cwd().parent
sys.path.append(str(BASE_DIR))

output_path = BASE_DIR / "results" / "Scale_up_output_MS.csv"
ecw_country = BASE_DIR / "results" / "ECWbyCountry.csv"
ecw_region = BASE_DIR / "results" / "ECWbyRegion.csv"

In [10]:
# ---------- Core Functions ----------

def aggregate_by_region(df, region_type="UNregion"):
    """
    Aggregate countries into regions using country_converter.
    region_type can be 'UNregion', 'UNcontinent', etc.
    """
    cc = coco.CountryConverter()
    df = df.copy().reset_index().rename(columns={"index": "Country"})
    df["Region"] = cc.convert(names=df["Country"].tolist(), to=region_type)
    df = df[df["Region"] != "not found"]

    region_data = {}
    for region, group in df.groupby("Region"):
        numeric = group.drop(columns=["Country", "Region"])
        summed = numeric.apply(lambda col: sum(col), axis=0)  # ufloat sums
        region_data[region] = summed

    region_df = pd.DataFrame(region_data).T
    region_df = region_df.reset_index().rename(columns={"index": "Region"})
    return region_df

def get_country_series(df, country):
    """Return time (weeks), nominal values, and uncertainties for a single country."""
    if country not in df.index:
        raise ValueError(f"{country} not found in DataFrame index")

    row = df.loc[country]  # Series of ufloats
    time = np.arange(len(row))
    values = np.array([nominal_value(v) for v in row])
    errors = np.array([std_dev(v) for v in row])
    return time, values, errors


def plot_country(df, country, ax=None, sigma=1):
    """Plot a single country's series with ±sigma bands."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(10,6))
    
    t, vals, errs = get_country_series(df, country)
    ax.plot(t, vals, label=country)
    ax.fill_between(t, vals - sigma*errs, vals + sigma*errs, alpha=0.3)
    ax.set_xlabel("Weeks")
    ax.set_ylabel("Value")
    ax.legend()
    return ax


def plot_multiple_countries(df, countries, sigma=1):
    """Plot multiple countries together."""
    fig, ax = plt.subplots(figsize=(12,8))
    for c in countries:
        plot_country(df, c, ax=ax, sigma=sigma)
    plt.title(f"Country series with ±{sigma}σ bands")
    plt.show()


def plot_region(df, df_ecw, region, length=None, CADRPP=100, ax=None, sigma=1):
    """Plot aggregated UN region time series with uncertainty."""
    if region not in df["Region"].values:
        raise ValueError(f"{region} not found in DataFrame")
    row = df[df["Region"] == region].iloc[0, 1:]  # drop "Region"
    t = np.arange(len(row))
    vals = np.array([nominal_value(v) for v in row])
    errs = np.array([std_dev(v) for v in row])
    
    if length is not None:
        vals = vals[0:length]
        t = t[0:length]
        errs = errs[0:length]
            
    if ax is None:
        fig, ax = plt.subplots(figsize=(10,6))
    
    # Get ECW values first to determine color
    ecw_lower, ecw_upper = get_ECW_region(df_ecw, region)
    CADR_lower = CADRPP*ecw_lower
    CADR_upper = CADRPP*ecw_upper
    
    # Plot the main line and get its color
    line = ax.plot(t, vals, label=region)[0]
    line_color = line.get_color()
    
    # Add horizontal shaded bands FIRST (so they're in the background)
    ax.axhspan(CADR_lower, CADR_upper, color=line_color, alpha=0.15, 
               label=f'CADR Range ({CADR_lower:.1f}-{CADR_upper:.1f})')
    
    # Add uncertainty fill on top (will be darker where they overlap)
    ax.fill_between(t, vals - sigma*errs, vals + sigma*errs, 
                    alpha=0.3, color=line_color, label=f'{sigma}σ uncertainty')
    
    # Add horizontal dashed lines for the bounds
    ax.axhline(y=CADR_lower, color=line_color, linestyle='--', alpha=0.8, linewidth=1)
    ax.axhline(y=CADR_upper, color=line_color, linestyle='--', alpha=0.8, linewidth=1)
    
    ax.set_xlabel("Weeks")
    ax.set_ylabel("Value")
    ax.legend()
    return ax


def plot_multiple_regions(df, df_ecw, regions, length=None, CADRPP=100, sigma=1):
    """Plot multiple regions together."""
    fig, ax = plt.subplots(figsize=(12,8))
    for r in regions:
        plot_region(df, df_ecw, r, length=length, CADRPP=CADRPP,ax=ax, sigma=sigma)
    plt.title(f"Region series, CADRPP: {CADRPP} L/s/pp")
    plt.show()

def get_ECW_region(df_ecw, region):
    row = df_ecw[df_ecw["Region"] == region]
    ECW_UPPER = int(row["ECW ILO"].iloc[0])
    ECW_Lower = int(row["ECW Poll"].iloc[0])
    return ECW_Lower, ECW_UPPER

In [15]:
## ---------------------------- Scale Up Plotting for Tester ---------------------------- 
df_loaded = pd.read_pickle(output_path.with_suffix(".pkl"))

region_df = aggregate_by_region(df_loaded, region_type="UNregion")
Region_list = region_df['Region'].tolist()
#print(region_df)


# plot_country(df_loaded, "Global", sigma=2)
#plot_multiple_countries(df_loaded, ["China", "India", "Japan"], sigma=2)

# Region-level plots


CADRPP = 50 # L/s


df_ecw = pd.read_csv(ecw_region)




plot_multiple_regions(region_df, df_ecw,  ["Eastern Asia", "Northern America", "Northern Europe"],length=52, CADRPP=CADRPP, sigma =2)
['Australia and New Zealand']


# for i in Region_list:
#     plot_region(region_df, df_ecw, i, length=52, CADRPP=CADRPP, ax=None, sigma=1)


AttributeError: Can't get attribute 'CallableStdDev' on <module 'uncertainties.core' from '/Users/mitchellscott/opt/anaconda3/envs/test_env/lib/python3.10/site-packages/uncertainties/core.py'>